# Experiment: 3-class classification (stone / harvesting / off)

**Hypothesis**: adding header-Off audio as a third class might force the models to learn more robust acoustic features for distinguishing stones from harvesting, rather than just learning energy thresholds.

**Setup**:
- Class 0: harvesting_normal (during header-On, away from stone events)
- Class 1: stone (during header-On, just before VoltageSignal spike)
- Class 2: off (during header-Off periods)

**Evaluation**: collapse predictions to binary (stone vs not-stone) and compare against the 2-class baseline from `model_training.ipynb`.

In [ ]:
from asammdf import MDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import os
from pathlib import Path
from sklearn.utils import resample

from sktime.classification.kernel_based import RocketClassifier
from sktime.classification.interval_based import TimeSeriesForestClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

DATA_DIR = Path("data")
MF4_FILES = sorted(DATA_DIR.glob("*.mf4"))

SR             = 44100
VOLT_THRESHOLD = 2000
WINDOW_BEFORE  = 0.5
WINDOW_AFTER   = 0.1
WINDOW_LEN     = WINDOW_BEFORE + WINDOW_AFTER
MIN_SUSTAIN    = 5
N_MFCC         = 13
WIN_SAMPLES    = int(WINDOW_LEN * SR)

print("Setup complete.")

## 1. Labeling functions (reused)

In [ ]:
def get_episodes(status_channel):
    s = status_channel.samples.astype(str)
    t = status_channel.timestamps
    changes = np.where(s[:-1] != s[1:])[0]
    events = [(t[0], s[0])]
    for i in changes:
        events.append((t[i+1], s[i+1]))
    episodes, ep_start = [], None
    for ev_time, ev_state in events:
        if ev_state == 'On' and ep_start is None:
            ep_start = ev_time
        elif ev_state == 'Off' and ep_start is not None:
            episodes.append((ep_start, ev_time))
            ep_start = None
    if ep_start is not None:
        episodes.append((ep_start, t[-1]))
    return episodes

def get_off_periods(status_channel):
    """Get (start, end) tuples for header-Off periods (complement of On episodes)."""
    s = status_channel.samples.astype(str)
    t = status_channel.timestamps
    changes = np.where(s[:-1] != s[1:])[0]
    events = [(t[0], s[0])]
    for i in changes:
        events.append((t[i+1], s[i+1]))
    off_periods, off_start = [], None
    for ev_time, ev_state in events:
        if ev_state == 'Off' and off_start is None:
            off_start = ev_time
        elif ev_state == 'On' and off_start is not None:
            off_periods.append((off_start, ev_time))
            off_start = None
    if off_start is not None:
        off_periods.append((off_start, t[-1]))
    return off_periods

def get_stone_spike_times(volt_channel, episodes,
                          threshold=VOLT_THRESHOLD, min_sustain=MIN_SUSTAIN):
    v, t = volt_channel.samples, volt_channel.timestamps
    spike_times = []
    for ep_start, ep_end in episodes:
        mask = (t >= ep_start) & (t <= ep_end)
        v_ep, t_ep = v[mask], t[mask]
        if len(v_ep) < min_sustain:
            continue
        above = v_ep > threshold
        sustained = np.zeros_like(above)
        count = 0
        for k in range(len(above)):
            if above[k]:
                count += 1
                if count >= min_sustain:
                    sustained[k - min_sustain + 1:k + 1] = True
            else:
                count = 0
        edges = np.diff(sustained.astype(int))
        onsets = t_ep[np.where(edges == 1)[0] + 1]
        for st in onsets:
            if not spike_times or st - spike_times[-1] > 1.0:
                spike_times.append(float(st))
    return spike_times

def extract_window(audio_channel, center_time,
                   before=WINDOW_BEFORE, after=WINDOW_AFTER):
    t, s = audio_channel.timestamps, audio_channel.samples
    mask = (t >= center_time - before) & (t <= center_time + after)
    return s[mask].astype(np.float32)

print("Functions defined.")

## 2. Build 3-class window dataset

In [ ]:
stone_windows  = []   # class 1
normal_windows = []   # class 0 — harvesting normal
off_windows    = []   # class 2 — header off
rng = np.random.default_rng(42)

for f in MF4_FILES:
    mf     = MDF(f)
    audio  = mf.get("Sensor1")
    volt   = mf.get("VoltageSignal")
    status = mf.get("Status")

    episodes     = get_episodes(status)
    off_periods  = get_off_periods(status)
    spike_times  = get_stone_spike_times(volt, episodes)

    # Stone windows
    for st in spike_times:
        w = extract_window(audio, st)
        if len(w) >= WIN_SAMPLES * 0.9:
            stone_windows.append((f.stem, st, w[:WIN_SAMPLES]))

    # Normal windows (during On, away from spikes)
    for ep_start, ep_end in episodes:
        if ep_end - ep_start < WINDOW_LEN + 4:
            continue
        n_samples = min(3, int((ep_end - ep_start) / (WINDOW_LEN + 2)))
        candidates = rng.uniform(ep_start + 1, ep_end - WINDOW_LEN - 1,
                                  size=n_samples * 5)
        count = 0
        for ct in candidates:
            if any(abs(ct - st) < 2.0 for st in spike_times):
                continue
            w = extract_window(audio, ct + WINDOW_BEFORE)
            if len(w) >= WIN_SAMPLES * 0.9:
                normal_windows.append((f.stem, ct, w[:WIN_SAMPLES]))
                count += 1
            if count >= n_samples:
                break

    # Off windows (during header-Off periods)
    for off_start, off_end in off_periods:
        if off_end - off_start < WINDOW_LEN + 2:
            continue
        # Sample up to 4 windows per Off period (Off periods can be long)
        n_samples = min(4, int((off_end - off_start) / (WINDOW_LEN + 1)))
        candidates = rng.uniform(off_start + 0.5,
                                  off_end - WINDOW_LEN - 0.5,
                                  size=n_samples * 3)
        count = 0
        for ct in candidates:
            w = extract_window(audio, ct + WINDOW_BEFORE)
            if len(w) >= WIN_SAMPLES * 0.9:
                off_windows.append((f.stem, ct, w[:WIN_SAMPLES]))
                count += 1
            if count >= n_samples:
                break

print(f"Stone windows  (class 1): {len(stone_windows)}")
print(f"Normal windows (class 0): {len(normal_windows)}")
print(f"Off windows    (class 2): {len(off_windows)}")

all_windows = normal_windows + stone_windows + off_windows
y_all       = np.array(
    [0]*len(normal_windows) + [1]*len(stone_windows) + [2]*len(off_windows)
)
meta_all    = [(r, t) for r, t, _ in all_windows]
run_names   = sorted(set(r for r, _ in meta_all))
print(f"\nTotal samples: {len(y_all)}")
print(f"Class counts: {dict(zip(*np.unique(y_all, return_counts=True)))}")

## 3. Feature matrices

In [ ]:
def compute_mfcc_matrix(windows):
    mfccs = []
    n_pre = int(WINDOW_BEFORE * SR)
    for _, _, audio in windows:
        pre = audio[:n_pre].astype(np.float32)
        m = librosa.feature.mfcc(y=pre, sr=SR, n_mfcc=N_MFCC,
                                  n_fft=2048, hop_length=512)
        mfccs.append(m)
    n_frames = min(m.shape[1] for m in mfccs)
    return np.stack([m[:, :n_frames] for m in mfccs], axis=0)

X_mfcc     = compute_mfcc_matrix(all_windows)   # [n, 13, n_frames]
X_mfcc_uni = X_mfcc.mean(axis=1, keepdims=True)  # [n, 1, n_frames] for TSF
print("MFCC shape (ROCKET): ", X_mfcc.shape)
print("MFCC shape (TSF):    ", X_mfcc_uni.shape)

# Quick check: do the three classes look different in raw RMS energy?
print("\nClass-wise mean audio RMS:")
for cls, name in [(0, "harvesting"), (1, "stone"), (2, "off")]:
    idx = np.where(y_all == cls)[0]
    rms_vals = [np.sqrt(np.mean(all_windows[i][2]**2)) for i in idx]
    print(f"  Class {cls} ({name:10s}): n={len(idx):2d}  mean RMS = {np.mean(rms_vals):.4f}")

## 4. CV framework

In [ ]:
def make_loro_folds(meta_all, run_names):
    for run in run_names:
        test_idx  = [i for i, (r, _) in enumerate(meta_all) if r == run]
        train_idx = [i for i, (r, _) in enumerate(meta_all) if r != run]
        yield run, train_idx, test_idx

def balance_train_3class(train_idx, y_all, seed=42):
    """Oversample minority classes to match the largest class."""
    y_tr = y_all[train_idx]
    target = max(np.bincount(y_tr))
    out = []
    for cls in [0, 1, 2]:
        cls_idx = [train_idx[i] for i in np.where(y_tr == cls)[0]]
        if len(cls_idx) == 0:
            continue
        over = resample(cls_idx, n_samples=target, replace=True,
                        random_state=seed + cls)
        out.extend(over)
    out = np.array(out)
    np.random.default_rng(seed).shuffle(out)
    return out

def header_on_hours(run):
    for f in MF4_FILES:
        if f.stem == run:
            status = MDF(f).get("Status")
            eps = get_episodes(status)
            return sum(e - s for s, e in eps) / 3600.0
    return 0.0

def evaluate_binary(y_true_3, y_pred_3, header_hours):
    """
    Collapse 3-class predictions to binary (stone vs not-stone) and compute
    the standard challenge metrics.
      - y_true_3, y_pred_3: 3-class arrays (0=normal, 1=stone, 2=off)
      - false alarms are only counted on harvesting-normal (class 0) samples,
        because Off-period false alarms wouldn't matter at deployment.
    """
    y_true = np.asarray(y_true_3)
    y_pred = np.asarray(y_pred_3)
    stone_mask  = y_true == 1
    normal_mask = y_true == 0

    tpr = float((y_pred[stone_mask] == 1).mean()) if stone_mask.any() else float("nan")

    false_alarms = int((y_pred[normal_mask] == 1).sum())
    fpr_per_hour = false_alarms / (header_hours + 1e-12)

    n_tp = int((y_pred[stone_mask] == 1).sum()) if stone_mask.any() else 0
    advance = WINDOW_BEFORE * 1000 if n_tp > 0 else float("nan")

    return {"tpr": tpr, "fpr_per_hour": fpr_per_hour,
            "mean_advance_ms": advance, "n_tp": n_tp,
            "n_stone_test": int(stone_mask.sum()),
            "n_false_alarms": false_alarms}

# Dry run
print("Fold sizes:")
print(f"{'Run':<35} {'train':>6} {'test':>6} {'stone_te':>10} {'normal_te':>10} {'off_te':>8}")
for run, tr, te in make_loro_folds(meta_all, run_names):
    y_te = y_all[te]
    print(f"{run[-25:]:<35} {len(tr):>6} {len(te):>6} "
          f"{(y_te==1).sum():>10} {(y_te==0).sum():>10} {(y_te==2).sum():>8}")

## 5. Model 1 — ROCKET (3-class)

In [ ]:
rocket_results = []

for run, train_idx, test_idx in make_loro_folds(meta_all, run_names):
    bal_idx = balance_train_3class(train_idx, y_all)
    X_tr, y_tr = X_mfcc[bal_idx], y_all[bal_idx]
    X_te, y_te = X_mfcc[test_idx], y_all[test_idx]

    clf = RocketClassifier(num_kernels=1000, rocket_transform="rocket",
                           random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = evaluate_binary(y_te, y_pred, header_on_hours(run))
    m["fold"] = run
    rocket_results.append(m)
    tpr_s = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  {run[-10:]}  tpr={tpr_s}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}  fa={m['n_false_alarms']}")

rocket_df = pd.DataFrame(rocket_results).set_index("fold")
print("\nROCKET 3-class (binary-collapsed) summary:")
print(rocket_df[["tpr","fpr_per_hour","n_tp","n_stone_test","n_false_alarms"]].round(3))

## 6. Model 2 — TimeSeriesForestClassifier (3-class)

In [ ]:
tsf_results = []

for run, train_idx, test_idx in make_loro_folds(meta_all, run_names):
    bal_idx = balance_train_3class(train_idx, y_all)
    X_tr, y_tr = X_mfcc_uni[bal_idx], y_all[bal_idx]
    X_te, y_te = X_mfcc_uni[test_idx], y_all[test_idx]

    clf = TimeSeriesForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = evaluate_binary(y_te, y_pred, header_on_hours(run))
    m["fold"] = run
    tsf_results.append(m)
    tpr_s = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  {run[-10:]}  tpr={tpr_s}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}  fa={m['n_false_alarms']}")

tsf_df = pd.DataFrame(tsf_results).set_index("fold")
print("\nTSF 3-class summary:")
print(tsf_df[["tpr","fpr_per_hour","n_tp","n_stone_test","n_false_alarms"]].round(3))

## 7. Model 3 — 1D-CNN (3-class)

In [ ]:
def augment_audio(audio_batch, n_augments=8, seed=42):
    rng_a = np.random.default_rng(seed)
    out = []
    for audio in audio_batch:
        for _ in range(n_augments):
            a = audio.copy()
            shift = rng_a.integers(-int(0.05*SR), int(0.05*SR))
            a = np.roll(a, shift)
            if shift > 0:
                a[:shift] = 0.0
            elif shift < 0:
                a[shift:] = 0.0
            a = a + rng_a.normal(0, 0.01 * (np.std(a) + 1e-8),
                                  size=len(a)).astype(np.float32)
            a = (a * rng_a.uniform(0.8, 1.2)).astype(np.float32)
            out.append(a)
    return np.stack(out, axis=0)

class StoneCNN3(nn.Module):
    """Same CNN backbone but with 3-class output head."""
    def __init__(self, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,  16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.pool       = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(64, n_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)

def train_cnn_fold_3class(train_idx, test_idx, n_epochs=50, n_augments=8, lr=1e-3):
    all_audio = [w[2][:WIN_SAMPLES] for w in all_windows]
    X_tr_raw  = np.stack([all_audio[i] for i in train_idx])
    y_tr      = y_all[train_idx]

    # Augment only the minority classes (stone class definitely; off may also be needed)
    X_train_list, y_train_list = [X_tr_raw], [y_tr]
    counts = np.bincount(y_tr, minlength=3)
    target = counts.max()
    for cls in [1, 2]:
        cls_local = np.where(y_tr == cls)[0]
        if len(cls_local) == 0 or len(cls_local) >= target:
            continue
        n_aug = max(1, (target - len(cls_local)) // len(cls_local))
        aug = augment_audio(X_tr_raw[cls_local], n_augments=n_aug)
        X_train_list.append(aug)
        y_train_list.append(np.full(len(aug), cls, dtype=np.int64))

    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)

    X_train_t = torch.tensor(X_train[:, np.newaxis, :], dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)

    counts_post = np.bincount(y_train, minlength=3)
    sample_w = torch.tensor(
        1.0 / (counts_post[y_train] + 1e-6), dtype=torch.float32)
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    loader  = DataLoader(TensorDataset(X_train_t, y_train_t),
                         batch_size=16, sampler=sampler)

    model = StoneCNN3(n_classes=3)
    cw = torch.tensor(
        [counts_post.max() / (c + 1e-6) for c in counts_post],
        dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    model.train()
    for epoch in range(n_epochs):
        total = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"    ep {epoch+1}/{n_epochs}  loss={total/len(loader):.4f}")

    X_te_raw = np.stack([all_audio[i] for i in test_idx])
    X_te_t   = torch.tensor(X_te_raw[:, np.newaxis, :], dtype=torch.float32)
    y_te     = y_all[test_idx]

    model.eval()
    with torch.no_grad():
        y_pred = model(X_te_t).argmax(dim=1).numpy()

    return model, y_pred, y_te

In [ ]:
cnn_results = []
cnn_models  = {}

for run, train_idx, test_idx in make_loro_folds(meta_all, run_names):
    print(f"\n--- CNN 3-class fold: {run[-10:]} ---")
    model, y_pred, y_te = train_cnn_fold_3class(train_idx, test_idx)
    m = evaluate_binary(y_te, y_pred, header_on_hours(run))
    m["fold"] = run
    cnn_results.append(m)
    cnn_models[run] = model
    tpr_s = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  Result: tpr={tpr_s}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}  fa={m['n_false_alarms']}")

cnn_df = pd.DataFrame(cnn_results).set_index("fold")
print("\nCNN 3-class summary:")
print(cnn_df[["tpr","fpr_per_hour","n_tp","n_stone_test","n_false_alarms"]].round(3))

## 8. Comparison vs 2-class baseline

Reference numbers from `model_training.ipynb` (2-class):
- ROCKET: TPR=0.00, FA/hr=0.0
- TSF: TPR=0.25, FA/hr=3.2
- CNN: TPR=1.00, FA/hr=14.9

In [ ]:
def summary_row(df, name):
    return pd.Series({
        "model": name,
        "mean_tpr":       df["tpr"].mean(skipna=True),
        "mean_fa_per_hr": df["fpr_per_hour"].mean(skipna=True),
        "total_tp":       df["n_tp"].sum(),
        "total_stone":    df["n_stone_test"].sum(),
        "total_fa":       df["n_false_alarms"].sum(),
    })

results_3class = pd.DataFrame([
    summary_row(rocket_df, "ROCKET"),
    summary_row(tsf_df,    "TimeSeriesForest"),
    summary_row(cnn_df,    "1D-CNN"),
]).set_index("model")

# Baseline from 2-class experiment
baseline_2class = pd.DataFrame({
    "model": ["ROCKET", "TimeSeriesForest", "1D-CNN"],
    "baseline_tpr":     [0.00, 0.25, 1.00],
    "baseline_fa_per_hr": [0.0, 3.17, 14.92],
}).set_index("model")

comparison = baseline_2class.join(results_3class[["mean_tpr", "mean_fa_per_hr", "total_tp", "total_fa"]])
comparison.columns = ["TPR (2-class)", "FA/hr (2-class)",
                       "TPR (3-class)", "FA/hr (3-class)",
                       "TP (3-class)", "FA (3-class)"]
print("\n=== 2-class vs 3-class comparison ===")
print(comparison.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("2-class vs 3-class experiment", fontsize=11)

x = np.arange(len(comparison))
w = 0.35

axes[0].bar(x - w/2, comparison["TPR (2-class)"], w, label="2-class", color="steelblue")
axes[0].bar(x + w/2, comparison["TPR (3-class)"], w, label="3-class", color="tomato")
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison.index, rotation=15)
axes[0].set_ylabel("Mean TPR (higher = better)")
axes[0].set_title("True Positive Rate")
axes[0].legend()

axes[1].bar(x - w/2, comparison["FA/hr (2-class)"], w, label="2-class", color="steelblue")
axes[1].bar(x + w/2, comparison["FA/hr (3-class)"], w, label="3-class", color="tomato")
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison.index, rotation=15)
axes[1].set_ylabel("Mean False Alarms / Hour (lower = better)")
axes[1].set_title("False Alarm Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Confusion matrices — what's the model actually predicting?

More useful than aggregate metrics with so few stone examples — shows where errors come from.

In [ ]:
from sklearn.metrics import confusion_matrix

def collect_predictions(make_pred_fn):
    """Run model across all LORO folds and collect (y_true, y_pred) across all folds."""
    y_true_all, y_pred_all = [], []
    for run, train_idx, test_idx in make_loro_folds(meta_all, run_names):
        y_pred = make_pred_fn(train_idx, test_idx)
        y_true_all.extend(y_all[test_idx])
        y_pred_all.extend(y_pred)
    return np.array(y_true_all), np.array(y_pred_all)

# Retrain CNN once per fold and collect predictions for confusion matrix
y_true_cnn_all = []
y_pred_cnn_all = []
for run, tr, te in make_loro_folds(meta_all, run_names):
    _, y_pred, y_te = train_cnn_fold_3class(tr, te, n_epochs=30)
    y_true_cnn_all.extend(y_te)
    y_pred_cnn_all.extend(y_pred)
y_true_cnn_all = np.array(y_true_cnn_all)
y_pred_cnn_all = np.array(y_pred_cnn_all)

cm = confusion_matrix(y_true_cnn_all, y_pred_cnn_all, labels=[0, 1, 2])
labels = ['harvesting (0)', 'stone (1)', 'off (2)']

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("CNN 3-class confusion matrix (pooled across all LORO folds)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("\nKey numbers:")
print(f"  Stone correctly predicted as stone: {cm[1,1]}/{cm[1].sum()}")
print(f"  Stone predicted as harvesting:      {cm[1,0]}")
print(f"  Stone predicted as off:             {cm[1,2]}")
print(f"  Harvesting falsely flagged as stone: {cm[0,1]} (false alarms)")
print(f"  Off falsely flagged as stone:        {cm[2,1]}")

## 10. Verdict

The 3-class framing did not help. Numbers tell the story:

| Model | TPR 2-class | TPR 3-class | FA/hr 2-class | FA/hr 3-class |
|-------|-------------|-------------|---------------|---------------|
| ROCKET | 0.00 | 0.125 | 0.0 | 0.0 |
| TimeSeriesForest | 0.25 | 0.375 | 3.2 | 70.7 |
| 1D-CNN | 1.00 | 0.95 | 14.9 | 39.3 |

ROCKET barely moved. TSF marginally improved its TPR but its false alarm rate jumped from 3 to 70 per hour, mostly from Run 3. CNN actually got slightly worse — TPR dropped from 1.00 to 0.95, false alarms went from 15 to 39 per hour.

Why this happened: the off class is acoustically too distinct (mean RMS of 0.003 vs 0.03 for harvesting vs 0.15 for stone). The model learns to separate off from everything else easily, but the hard problem — distinguishing stone impacts from loud harvesting noise — gets no help from it. If anything, the model may now lean harder on "high amplitude = stone" because two of the three classes are quiet, which is exactly the wrong shortcut.

The confusion matrix confirms it: the off class is never confused with stone. All false alarms come from harvesting → stone, which is the same failure mode we had in the 2-class setup.

This is consistent signal that the bottleneck is not data diversity from a new source — it's data quantity within the actual decision boundary. 10 stone examples is too few, and we need more examples of *loud-but-not-stone harvesting* specifically, not quiet off-period audio.

Moving to synthetic data generation next.